## Objectives
1. **Feature Scaling**: Understand why scaling via `StandardScaler` is required before calculating distance matrices.
2. **Elbow Curve (Inertia)**: Plot Within-Cluster Sum of Squares (WCSS) across $K \in [2, 10]$ to locate the inflection ("elbow") point.
3. **Silhouette Score**: Compute average silhouette coefficients to measure how well-separated each cluster is from neighboring clusters.
4. **Diagnostic Comparison**: Learn how to combine Inertia elbow curves and Silhouette peaks to choose an optimal $K$.

## Metrics Overview
* **Inertia**:
  $$\text{Inertia} = \sum_{i=1}^{K} \sum_{x \in C_i} \|x - \mu_i\|^2$$
* **Silhouette Coefficient** (for point $i$):
  $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
  Where $a(i)$ is mean intra-cluster distance, and $b(i)$ is mean nearest-cluster distance.

In [ ]:
'''
Inertia : General meaning is the tendency of an object to resist changes in its state of motion (newton's first law of motion).
Inertia in metrics : It is a measure of how well the data points are clustered around the centroids. Lower inertia indicates that the clusters are more compact and well-separated.

Inertia measures the internal cohesiveness of clusters by calculating the sum of squared distances between each data point and its assigned cluster centroid. Lower inertia indicates tighter and more compact clusters.

Inertia uses euclidean distance to measure the distance between each data point and its assigned cluster centroid.
'''

'''
Silhoutte : It is a metric used to evaluate the quality of clusters.
it not only considers the compactness of clusters but also the separation between clusters. Higher silhouette score indicates better-defined clusters.
'''
print()


| Feature | Inertia (WCSS) | Silhouette Coefficient |
| :--- | :--- | :--- |
| **What it measures** | Compactness (how tight clusters are) | Compactness and separation (boundaries) |
| **Value Range** | $0$ to $\infty$ (No upper bound) | $-1$ to $+1$ (Strictly bounded) |
| **Behavior with $K$** | Always decreases as $K$ increases (hard to use for picking $K$ alone) | Peaks at the optimal $K$, then drops if you over-cluster |
| **Interpretation** | Lower is better | Higher is better |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# step 1 Load the data & Preprocess Features
data_path = Path("./dataset/customer_segmentation.csv")
df = pd.read_csv(data_path)
# print(df.shape)
print(df.head())
print("All Features ", list(df.columns))
# select all numeric features for clustering
features = df.select_dtypes(include=[np.number]).columns
# drop customerID if present
if 'customer_id' in features:
    features = features.drop('customer_id')
print("Selected Features ", list(features))
# standardize the features
# standardize the features to have mean 0 and variance 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
print("Shape of standardized features: ", X_scaled.shape)
print("First 5 rows of standardized features: ")
print(X_scaled[:5])

## 2. Sweep Across Cluster Range $K \in [2, 10]$

In [16]:
k_range = range(2, 11)
inertias = []
silhouette_scores = []
for k in k_range:
    '''
    KMeans Parameters used
    n_clusters = k : number of clusters to form
    random_state = 42 : seed for random number generator
    n_init = 10 : number of time the k-means algorithm will be run with different centroid seeds
    init = 'k-means++' : method for initialization : other option is 'random'
    random_state = 42
    n_init = 10
    init = 'k-means++'
    '''
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10,init='k-means++')
    cluster_labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(sil_score)
    print(f"k = {k}, Inertia = {kmeans.inertia_:.2f}, Silhouette Score = {sil_score:.4f}")


k = 2, Inertia = 89.06, Silhouette Score = 0.4566
k = 3, Inertia = 65.88, Silhouette Score = 0.3758
k = 4, Inertia = 49.70, Silhouette Score = 0.3555
k = 5, Inertia = 39.72, Silhouette Score = 0.3490
k = 6, Inertia = 33.82, Silhouette Score = 0.3462
k = 7, Inertia = 29.79, Silhouette Score = 0.3351
k = 8, Inertia = 25.21, Silhouette Score = 0.3404
k = 9, Inertia = 22.29, Silhouette Score = 0.3432
k = 10, Inertia = 20.22, Silhouette Score = 0.3307


## 3. Visualize Diagnostic Curves